# NSE XBRL reader by filing format

Read BANKING, NBFC, INDAS, and NONINDAS instance XML files; inspect actual tags, contexts,
units, and periods; export financial records with their source evidence.

**Run all cells in order.** Upload XML files, or ZIP archives containing XML files. Alternatively,
mount a Drive folder. ZIP entries are read in memory without extracting paths to disk.
Source files are not changed. This is a targeted financial extractor, not a complete XBRL taxonomy validator.
Inline XBRL/HTML and unrecognized formats are reported rather than interpreted as known NSE formats.

**Important:** some inspected NSE files refer to `OneD`/`FourD` without defining those contexts.
Their financial values are retained in the inspection output, but are excluded from strict quarterly candidates.
The notebook never guesses a quarter from a context ID, filename date, or "Half yearly" label.
Missing context definitions require source repair or a separately verified reconstruction process.

The `decimals` attribute expresses accuracy, not a scaling multiplier. Numeric instance values are retained
in their declared units; "Lakhs" or "Crores" labels are not applied again.
See the [XBRL 2.1 specification](https://www.xbrl.org/specification/xbrl-2.1/rec-2003-12-31/xbrl-2.1-rec-2003-12-31+corrected-errata-2013-02-20.html)
and [Colab file input examples](https://colab.research.google.com/notebooks/io.ipynb).


In [ ]:
import pandas as pd
from IPython.display import display
import zipfile
import io
from pathlib import Path

INPUT_MODE = 'drive'  # 'upload' or 'drive'
DRIVE_FOLDER = '/content/drive/MyDrive/nse_xml_2022_2024'
SELECTED_BASIS = 'Standalone'  # 'Standalone', 'Consolidated', or 'All'
START_DATE = '2022-04-01'  # Set both dates to None to inspect all periods.
END_DATE = '2024-12-31'
MAX_FILES = 50000
MAX_FILE_BYTES = 25 * 1024 * 1024
MAX_TOTAL_BYTES = 500 * 1024 * 1024
OUTPUT_DIR = Path('/content/drive/MyDrive/nse-financial-research/data/interim')
pd.set_option('display.max_columns', 18)


## 1. Parser and format-specific mappings

PAT uses total profit/loss for the period: `ProfitLossForPeriod` / `ProfitLossForThePeriod`.
Continuing-operations profit, comprehensive income, and profit attributable only to owners are not
substituted. Banking operating profit before provisions is kept as a distinct measure.
Unknown namespaces, conflicting facts, nil values, and missing unit/context definitions remain flagged.
To support new taxonomy versions, review raw facts before adding exact aliases in `FORMAT_MAPS`.


In [ ]:
"""Conservative NSE XBRL instance extraction; embedded into the Colab notebook."""
import io
import json
import re
import xml.etree.ElementTree as ET
from datetime import date
from decimal import Decimal, InvalidOperation
from pathlib import Path

XBRLI = 'http://www.xbrl.org/2003/instance'
XSI = 'http://www.w3.org/2001/XMLSchema-instance'
XLINK = 'http://www.w3.org/1999/xlink'
KNOWN_FORMATS = {'INDAS', 'NBFC', 'BANKING', 'NONINDAS'}
BASE_MAP = {
    'PAT': ['ProfitLossForPeriod', 'ProfitLossForThePeriod'],
    'PAT_CONTINUING_OPERATIONS': ['ProfitLossForPeriodFromContinuingOperations', 'ProfitLossForThePeriodFromContinuingOperations'],
    'PAT_DISCONTINUED_OPERATIONS': ['ProfitLossFromDiscontinuedOperationsAfterTax', 'ProfitLossFromDiscontinuingOperationAfterTax'],
    'PAT_OWNERS_OF_PARENT': ['ProfitOrLossAttributableToOwnersOfParent'],
    'PAT_NONCONTROLLING_INTERESTS': ['ProfitOrLossAttributableToNonControllingInterests'],
    'REVENUE': ['RevenueFromOperations'],
    'TOTAL_INCOME': ['Income'],
    'OTHER_INCOME': ['OtherIncome'],
    'PROFIT_BEFORE_TAX': ['ProfitBeforeTax'],
    'TAX_EXPENSE': ['TaxExpense'],
    'EXPENSES': ['Expenses'],
    'FINANCE_COSTS': ['FinanceCosts'],
    'EPS_BASIC': ['BasicEarningsLossPerShareFromContinuingAndDiscontinuedOperations'],
    'EPS_DILUTED': ['DilutedEarningsLossPerShareFromContinuingAndDiscontinuedOperations'],
    'ASSETS': ['Assets'], 'LIABILITIES': ['Liabilities'], 'EQUITY': ['Equity'],
    'BORROWINGS': ['Borrowings'],
}
FORMAT_MAPS = {kind: {k: list(v) for k, v in BASE_MAP.items()} for kind in KNOWN_FORMATS}
FORMAT_MAPS['NONINDAS']['TOTAL_INCOME'] = ['Revenue']
FORMAT_MAPS['BANKING'].update({
    'PROFIT_BEFORE_TAX': ['ProfitLossFromOrdinaryActivitiesBeforeTax'],
    'OPERATING_PROFIT_BEFORE_PROVISIONS': ['OperatingProfitBeforeProvisionAndContingencies'],
    'INTEREST_EARNED': ['InterestEarned'], 'INTEREST_EXPENDED': ['InterestExpended'],
    'PROVISIONS': ['ProvisionsOtherThanTaxAndContingencies'],
    'EPS_BASIC': ['BasicEarningsPerShareAfterExtraordinaryItems'],
    'EPS_DILUTED': ['DilutedEarningsPerShareAfterExtraordinaryItems'],
    'GROSS_NPA': ['GrossNonPerformingAssets'], 'NET_NPA': ['NonPerformingAssets'],
})
METADATA_MAP = {
    'NSE_SYMBOL': ['Symbol'], 'COMPANY_NAME': ['NameOfCompany', 'NameOfBank'],
    'STANDALONE_CONSOLIDATED': ['NatureOfReportStandaloneConsolidated'],
    'AUDITED_FLAG': ['WhetherResultsAreAuditedOrUnaudited'],
    'REPORTED_QUARTER_LABEL': ['ReportingQuarter'],
    'REPORTING_CURRENCY': ['DescriptionOfPresentationCurrency'],
    'ROUNDING': ['LevelOfRoundingUsedInFinancialStatements'],
    'REPORTED_START': ['DateOfStartOfReportingPeriod'],
    'REPORTED_END': ['DateOfEndOfReportingPeriod'],
    'FINANCIAL_YEAR_START': ['DateOfStartOfFinancialYear'],
    'FINANCIAL_YEAR_END': ['DateOfEndOfFinancialYear'],
}


def local_name(tag):
    return tag.rsplit('}', 1)[-1]


def namespace(tag):
    return tag[1:].split('}', 1)[0] if tag.startswith('{') else ''


def as_number(text):
    try:
        value = Decimal(text.strip())
        return value if value.is_finite() else None
    except (InvalidOperation, AttributeError):
        return None


def safe_date(text):
    try:
        return date.fromisoformat(text[:10])
    except (TypeError, ValueError):
        return None


def detect_format(filename, root):
    prefix = re.split(r'[_\-.]', Path(filename).name.upper())[0]
    named = prefix if prefix in KNOWN_FORMATS else None
    concepts = {local_name(e.tag) for e in root.iter() if e.get('contextRef')}
    if 'OperatingProfitBeforeProvisionAndContingencies' in concepts:
        content = 'BANKING'
    elif 'ProfitLossForPeriodBeforeMinorityInterest' in concepts:
        content = 'NONINDAS'
    elif {'Loans', 'ImpairmentOnFinancialInstruments'} <= concepts:
        content = 'NBFC'
    elif 'ProfitLossForPeriod' in concepts:
        content = 'INDAS_OR_NBFC'
    else:
        content = 'UNKNOWN'
    conflict = bool(named and content in KNOWN_FORMATS and named != content)
    detected = 'UNKNOWN' if conflict else named or (content if content in KNOWN_FORMATS else 'UNKNOWN')
    return detected, f'filename={named or "unknown"}; content={content}', conflict


def parse_xbrl(filename, payload):
    # Preserve evidence rather than resolving ambiguous facts by document order.
    if re.search(br'<!\s*(?:DOCTYPE|ENTITY)\b', payload, flags=re.I):
        raise ValueError('DTD/entity declarations are unsupported.')
    ns = dict(pair for _, pair in ET.iterparse(io.BytesIO(payload), events=['start-ns']))
    root = ET.fromstring(payload)
    if root.tag != f'{{{XBRLI}}}xbrl':
        raise ValueError('Expected an XBRL instance XML. Inline XBRL/HTML is not supported.')
    def expand_qname(text):
        prefix, sep, name = (text or '').partition(':')
        return f'{{{ns.get(prefix, prefix)}}}{name}' if sep else f'{{{ns.get("", "")}}}{prefix}'
    fmt, evidence, conflict = detect_format(filename, root)
    issues = []
    def issue(kind, context='', field='', detail=''):
        issues.append({'SOURCE_FILE': filename, 'CONTEXT_ID': context,
                       'FIELD': field, 'ISSUE': kind, 'DETAIL': detail})
    if fmt == 'UNKNOWN':
        issue('UNKNOWN_OR_CONFLICTING_FORMAT', detail=evidence)
    contexts = {}
    for element in root.findall(f'{{{XBRLI}}}context'):
        cid = element.get('id')
        if cid in contexts:
            raise ValueError(f'Duplicate context ID: {cid}')
        def text_at(path):
            found = element.find(path)
            return found.text.strip() if found is not None and found.text else None
        dimensions = []
        for member in element.iter():
            if local_name(member.tag) in {'explicitMember', 'typedMember'}:
                dimensions.append({'dimension': member.get('dimension'),
                                   'member': ''.join(member.itertext()).strip()})
        containers = element.findall(f'.//{{{XBRLI}}}segment') + element.findall(f'.//{{{XBRLI}}}scenario')
        contexts[cid] = {
            'ENTITY_IDENTIFIER': text_at(f'{{{XBRLI}}}entity/{{{XBRLI}}}identifier'),
            'PERIOD_START': text_at(f'{{{XBRLI}}}period/{{{XBRLI}}}startDate'),
            'PERIOD_END': text_at(f'{{{XBRLI}}}period/{{{XBRLI}}}endDate'),
            'INSTANT': text_at(f'{{{XBRLI}}}period/{{{XBRLI}}}instant'),
            'DIMENSIONS': json.dumps(dimensions, sort_keys=True),
            'HAS_CONTEXT_DETAIL': any(len(c) > 0 for c in containers),
        }
    units = {}
    for element in root.findall(f'{{{XBRLI}}}unit'):
        uid = element.get('id')
        if uid in units:
            raise ValueError(f'Duplicate unit ID: {uid}')
        divide = element.find(f'{{{XBRLI}}}divide')
        if divide is None:
            numerator = [expand_qname(e.text) for e in element.findall(f'{{{XBRLI}}}measure')]
            denominator = []
        else:
            numerator = [expand_qname(e.text) for e in divide.findall(f'{{{XBRLI}}}unitNumerator/{{{XBRLI}}}measure')]
            denominator = [expand_qname(e.text) for e in divide.findall(f'{{{XBRLI}}}unitDenominator/{{{XBRLI}}}measure')]
        units[uid] = {'numerator': sorted(numerator), 'denominator': sorted(denominator)}
    facts = []
    for index, element in enumerate(root.iter()):
        cid = element.get('contextRef')
        if not cid:
            continue
        text = ''.join(element.itertext()).strip()
        nil = element.get(f'{{{XSI}}}nil', '').lower() in {'true', '1'}
        unit = element.get('unitRef')
        facts.append({'SOURCE_FILE': filename, 'FORMAT': fmt, 'FACT_INDEX': index,
                      'CONCEPT': local_name(element.tag), 'CONCEPT_QNAME': element.tag,
                      'NAMESPACE': namespace(element.tag), 'CONTEXT_ID': cid,
                      'RAW_VALUE': text, 'IS_NIL': nil, 'UNIT_REF': unit,
                      'UNIT_DEFINITION': json.dumps(units.get(unit), sort_keys=True),
                      'DECIMALS': element.get('decimals'),
                      'NUMERIC_VALUE': str(as_number(text)) if not nil and as_number(text) is not None else None,
                      'CONTEXT_DEFINED': cid in contexts})
    groups = {}
    for fact in facts:
        groups.setdefault(fact['CONTEXT_ID'], []).append(fact)
    rows = []
    known_namespace = lambda f: bool(re.search(r'https?://(?:www\.)?(?:bseindia|nseindia)\.com/', f['NAMESPACE']))
    def pick(group, aliases, cid, field, numeric=False):
        # Alias ordering expresses semantic preference, not a fuzzy tag match.
        for alias in aliases:
            matches = [f for f in group if f['CONCEPT'] == alias and not f['IS_NIL'] and known_namespace(f)]
            if not matches:
                continue
            comparable = {(f['NUMERIC_VALUE'] if numeric else f['RAW_VALUE'], f['CONCEPT_QNAME'],
                           f['UNIT_DEFINITION'] if numeric else '') for f in matches}
            if len(comparable) != 1:
                issue('CONFLICTING_FACTS', cid, field, alias)
                return None
            fact = matches[0]
            if numeric and fact['NUMERIC_VALUE'] is None:
                issue('NONNUMERIC_MAPPED_FACT', cid, field, fact['RAW_VALUE'])
                return None
            if len(matches) > 1:
                issue('IDENTICAL_DUPLICATE_FACTS', cid, field, alias)
            return fact
        return None
    for cid, group in groups.items():
        if not any(f['NUMERIC_VALUE'] is not None and f['UNIT_REF'] for f in group):
            continue
        ctx = contexts.get(cid)
        row = {'SOURCE_FILE': filename, 'FILING_FORMAT': fmt, 'FORMAT_EVIDENCE': evidence,
               'CONTEXT_ID': cid, 'CONTEXT_DEFINED': ctx is not None}
        row.update(ctx or {'ENTITY_IDENTIFIER': None, 'PERIOD_START': None, 'PERIOD_END': None,
                           'INSTANT': None, 'DIMENSIONS': 'UNKNOWN', 'HAS_CONTEXT_DETAIL': True})
        for field, aliases in METADATA_MAP.items():
            chosen = pick(group, aliases, cid, field)
            row[field] = chosen['RAW_VALUE'] if chosen else None
        # Only document-wide identity/currency/rounding are allowed to fall back.
        # Audit status and reporting basis are never borrowed from another period.
        for field in ['NSE_SYMBOL', 'COMPANY_NAME', 'REPORTING_CURRENCY', 'ROUNDING']:
            if row[field] is None:
                chosen = pick(facts, METADATA_MAP[field], cid, field)
                row[field] = chosen['RAW_VALUE'] if chosen else None
        for target, source in [('PERIOD_START', 'REPORTED_START'), ('PERIOD_END', 'REPORTED_END')]:
            if not row[target] and row[source]:
                row[target] = row[source]
                row[target + '_SOURCE'] = 'reporting-date fact; context missing or incomplete'
            elif row[target]:
                row[target + '_SOURCE'] = 'context'
            if row[target] and row[source] and row[target] != row[source]:
                issue('DATE_CONFLICT', cid, target, f"context={row[target]}; fact={row[source]}")
        if row['NSE_SYMBOL'] is None and ctx and ctx['ENTITY_IDENTIFIER']:
            # Identifier is retained separately; do not assume it is an NSE symbol.
            issue('SYMBOL_MISSING', cid, detail=ctx['ENTITY_IDENTIFIER'])
        if ctx is None:
            issue('UNDEFINED_CONTEXT', cid, detail='Values retained for inspection; excluded from strict quarterly output.')
        field_facts = {}
        for field, aliases in FORMAT_MAPS.get(fmt, {}).items():
            chosen = pick(group, aliases, cid, field, numeric=True)
            row[field] = None
            row[field + '_TAG'] = chosen['CONCEPT_QNAME'] if chosen else None
            row[field + '_UNIT'] = chosen['UNIT_DEFINITION'] if chosen else None
            row[field + '_DECIMALS'] = chosen['DECIMALS'] if chosen else None
            if chosen:
                expected_eps = field.startswith('EPS_')
                definition = units.get(chosen['UNIT_REF'])
                num = definition['numerator'] if definition else []
                den = definition['denominator'] if definition else []
                currency = (len(num) == 1 and re.fullmatch(r'\{http://www.xbrl.org/2003/iso4217\}[A-Z]{3}', num[0]))
                correct_unit = currency and ((den == [f'{{{XBRLI}}}shares']) if expected_eps else not den)
                if not correct_unit:
                    issue('UNSUPPORTED_OR_MISSING_UNIT', cid, field, chosen['UNIT_DEFINITION'])
                else:
                    row[field] = chosen['NUMERIC_VALUE']
                    field_facts[field] = chosen
        start, end = safe_date(row['PERIOD_START']), safe_date(row['PERIOD_END'])
        row['DURATION_DAYS'] = (end - start).days + 1 if start and end else None
        row['PERIOD_KIND'] = ('instant' if row['INSTANT'] else 'quarter_like' if row['DURATION_DAYS'] is not None
                              and 70 <= row['DURATION_DAYS'] <= 110 else 'duration_other_or_unknown')
        row['PAT_DEFINITION'] = 'Total profit/loss for period; not owners-only or comprehensive income'
        if row.get('PAT') is None and row.get('PROFIT_BEFORE_TAX') is not None:
            issue('TOTAL_PAT_UNAVAILABLE', cid, 'PAT', 'Review raw facts; no substitution from continuing operations or PBT minus tax.')
        if all(row.get(k) is not None for k in ['PROFIT_BEFORE_TAX', 'TAX_EXPENSE', 'PAT']):
            definitions = {row[k + '_UNIT'] for k in ['PROFIT_BEFORE_TAX', 'TAX_EXPENSE', 'PAT']}
            if len(definitions) == 1:
                difference = Decimal(row['PROFIT_BEFORE_TAX']) - Decimal(row['TAX_EXPENSE']) - Decimal(row['PAT'])
                row['PBT_MINUS_TAX_MINUS_PAT'] = str(difference)
                if difference:
                    issue('PAT_RECONCILIATION_DIFFERENCE', cid, 'PAT', str(difference))
        row['STRICT_QUARTER_CANDIDATE'] = bool(ctx is not None and not row['HAS_CONTEXT_DETAIL']
            and row['PERIOD_KIND'] == 'quarter_like' and fmt in KNOWN_FORMATS
            and (row['STANDALONE_CONSOLIDATED'] or '').strip().lower() in {'standalone', 'consolidated'}
            and row.get('PAT') is not None and row.get('NSE_SYMBOL'))
        if row['STRICT_QUARTER_CANDIDATE'] and any(i['CONTEXT_ID'] == cid and i['ISSUE'] in
            {'DATE_CONFLICT', 'CONFLICTING_FACTS'} for i in issues):
            row['STRICT_QUARTER_CANDIDATE'] = False
        rows.append(row)
    # Financial statement concepts that need mapping remain visible, never guessed.
    mapped = {a for aliases in FORMAT_MAPS.get(fmt, {}).values() for a in aliases}
    unmapped = sorted({f['CONCEPT'] for f in facts if f['NUMERIC_VALUE'] is not None and
                       any(t in f['CONCEPT'].lower() for t in ['profit', 'tax', 'revenue', 'earning'])
                       and f['CONCEPT'] not in mapped})
    summary = {'SOURCE_FILE': filename, 'FORMAT': fmt, 'FORMAT_EVIDENCE': evidence,
               'FACTS': len(facts), 'DEFINED_CONTEXTS': len(contexts), 'EXTRACTED_CONTEXT_ROWS': len(rows),
               'UNDEFINED_CONTEXT_REFS': len({f['CONTEXT_ID'] for f in facts if not f['CONTEXT_DEFINED']}),
               'PAT_CONTEXT_ROWS': sum(r.get('PAT') is not None for r in rows),
               'STRICT_QUARTER_CANDIDATES': sum(r['STRICT_QUARTER_CANDIDATE'] for r in rows),
               'UNMAPPED_RELATED_CONCEPTS': '|'.join(unmapped)}
    return rows, facts, issues, summary


## 2. Select XML files or ZIP archives


In [ ]:
input_entries = []
total_bytes = 0
def accept(name, payload):
    global total_bytes
    if len(input_entries) >= MAX_FILES:
        raise ValueError('File count limit exceeded. Process smaller batches.')
    if len(payload) > MAX_FILE_BYTES or total_bytes + len(payload) > MAX_TOTAL_BYTES:
        raise ValueError('Input size limit exceeded. Process smaller batches or adjust limits.')
    input_entries.append((name, payload))
    total_bytes += len(payload)
if INPUT_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    for name, payload in uploaded.items():
        if name.lower().endswith(('.xml', '.xbrl')):
            accept(name, payload)
        elif name.lower().endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(payload)) as archive:
                for item in archive.infolist():
                    if not item.is_dir() and item.filename.lower().endswith(('.xml', '.xbrl')):
                        if item.file_size > MAX_FILE_BYTES or total_bytes + item.file_size > MAX_TOTAL_BYTES:
                            raise ValueError('ZIP entry size limit exceeded. Use smaller batches.')
                        accept(name + '/' + item.filename, archive.read(item))
elif INPUT_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    for p in sorted(Path(DRIVE_FOLDER).rglob('*')):
        if p.is_file() and p.suffix.lower() in {'.xml', '.xbrl'}:
            if p.stat().st_size > MAX_FILE_BYTES:
                raise ValueError('File too large: ' + str(p))
            accept(str(p), p.read_bytes())
else:
    raise ValueError('Choose upload or drive.')
print('XML files selected:', len(input_entries))
if not input_entries:
    raise ValueError('No XML files found.')


## 3. Extract and inspect all reporting contexts

One filing can contain quarterly, cumulative, instant, and segment facts. They remain separate.
No first-context selection or automatic annual-to-quarter conversion is performed.
Raw facts preserve taxonomy names, declared units, and numeric text for tracing every extracted value.


In [ ]:
financial_rows, raw_facts, quality_issues, file_summaries, failures = [], [], [], [], []
for position, (name, payload) in enumerate(input_entries, 1):
    try:
        rows, facts, issues, summary = parse_xbrl(name, payload)
        financial_rows.extend(rows)
        raw_facts.extend(facts)
        quality_issues.extend(issues)
        file_summaries.append(summary)
    except (ValueError, ET.ParseError) as exc:
        failures.append({'SOURCE_FILE': name, 'ERROR': str(exc)})
    if position % 250 == 0:
        print(f'Processed {position:,}/{len(input_entries):,} files')
financial = pd.DataFrame(financial_rows)
facts = pd.DataFrame(raw_facts)
issues = pd.DataFrame(quality_issues, columns=['SOURCE_FILE', 'CONTEXT_ID', 'FIELD', 'ISSUE', 'DETAIL'])
summaries = pd.DataFrame(file_summaries)
errors = pd.DataFrame(failures, columns=['SOURCE_FILE', 'ERROR'])
display(summaries.head(20))
display(errors.head(20))
if financial.empty:
    raise ValueError('No financial context rows extracted. Inspect errors and format evidence.')
for field in ['PAT', 'REVENUE', 'TOTAL_INCOME', 'PROFIT_BEFORE_TAX', 'TAX_EXPENSE']:
    if field not in financial:
        financial[field] = pd.NA
print('Mapped values are numeric strings to preserve precision; original facts remain in raw_facts.csv.')
display(financial[['SOURCE_FILE', 'FILING_FORMAT', 'NSE_SYMBOL', 'CONTEXT_ID', 'CONTEXT_DEFINED',
                   'STANDALONE_CONSOLIDATED', 'PERIOD_START', 'PERIOD_END', 'DURATION_DAYS',
                   'PAT', 'PAT_TAG', 'STRICT_QUARTER_CANDIDATE']].head(25)
        if 'PAT_TAG' in financial else financial.head(25))


## 4. Scope checks and conservative quarterly export

Strict candidates require a defined, unsegmented context, a quarter-like duration, explicit reporting basis,
a known format, symbol, and PAT with a valid monetary unit. They are still candidates, not certified observations.
Basis reported only in a different context is not silently copied. Instant balance-sheet contexts remain separate.

Repeated company-period-basis rows are marked for review and excluded from the conservative output.
Do not resolve revisions by selecting the last filename: actual publication timestamps are required.


In [ ]:
financial['END_DATE_PARSED'] = pd.to_datetime(financial.PERIOD_END, errors='coerce')
basis = financial.STANDALONE_CONSOLIDATED.fillna('').str.strip().str.casefold()
scoped = financial.copy() if SELECTED_BASIS == 'All' else financial.loc[basis.eq(SELECTED_BASIS.casefold())].copy()
if START_DATE is not None:
    scoped = scoped.loc[scoped.END_DATE_PARSED.ge(pd.Timestamp(START_DATE))]
if END_DATE is not None:
    scoped = scoped.loc[scoped.END_DATE_PARSED.le(pd.Timestamp(END_DATE))]
strict = scoped.loc[scoped.STRICT_QUARTER_CANDIDATE].copy()
keys = ['NSE_SYMBOL', 'PERIOD_START', 'PERIOD_END', 'STANDALONE_CONSOLIDATED']
strict['DUPLICATE_REVIEW'] = strict.duplicated(keys, keep=False)
duplicates = strict.loc[strict.DUPLICATE_REVIEW].copy()
quarterly = strict.loc[~strict.DUPLICATE_REVIEW].copy()
print('All context rows:', len(financial), '| selected dates/basis:', len(scoped))
print('Strict candidates before duplicate review:', len(strict), '| conservative quarterly rows:', len(quarterly))
print('Missing-context rows are inspection-only and never included as strict candidates.')
format_profile = financial.groupby('FILING_FORMAT').agg(
    context_rows=('SOURCE_FILE', 'size'), files=('SOURCE_FILE', 'nunique'),
    company_symbols=('NSE_SYMBOL', 'nunique'), populated_pat=('PAT', 'count'),
    strict_candidates=('STRICT_QUARTER_CANDIDATE', 'sum')).reset_index()
display(format_profile)
display(issues.ISSUE.value_counts().rename_axis('issue').reset_index(name='count'))
keyword = 'Profit'  # Change to Revenue, Assets, Earnings, etc. for tag discovery.
display(facts.loc[facts.CONCEPT.str.contains(keyword, case=False, regex=False),
                  ['SOURCE_FILE', 'CONCEPT_QNAME', 'CONTEXT_ID', 'RAW_VALUE', 'UNIT_REF', 'DECIMALS']].head(30))


## 5. Download the outputs

- `all_financial_contexts.csv`: mapped values, including flagged contexts for inspection.
- `company_quarterly_financials_extracted.csv`: conservative standalone (or configured basis) candidates.
- `raw_facts.csv`: source concepts, values, units, and context references.
- `quality_issues.csv`, `duplicate_review.csv`, `file_summary.csv`, `format_profile.csv`, `errors.csv`.

The quarterly filename is compatible with the earlier scope-analysis notebook. Some previously used fields
may still be missing, and field definitions must be checked before combining years or building targets.
No industry classification, availability dates, or missing financial values are fabricated.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export_tables = {'all_financial_contexts.csv': financial,
                 'company_quarterly_financials_extracted.csv': quarterly,
                 'raw_facts.csv': facts, 'quality_issues.csv': issues,
                 'duplicate_review.csv': duplicates, 'file_summary.csv': summaries,
                 'format_profile.csv': format_profile, 'errors.csv': errors}
for name, table in export_tables.items():
    table.to_csv(OUTPUT_DIR / name, index=False)
archive_path = OUTPUT_DIR.parent / 'xbrl_format_extraction_results.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in export_tables:
        archive.write(OUTPUT_DIR / name, arcname=name)
print('Saved:', archive_path)
if INPUT_MODE in {'upload', 'drive'}:
    from google.colab import files
    files.download(str(archive_path))
